# Waste Classification - Model Training & Comparison

Full pipeline:
1. **Load & explore** the preprocessed dataset (`.npz`)
2. **Feature extraction** with EfficientNetV2-S (pre-trained neural network)
3. **Train & compare** 4 sklearn models (Random Forest, XGBoost, SVM, LightGBM)
4. **Cross-validation** (StratifiedKFold) to evaluate robustness
5. **Hyperparameter optimization** with RandomizedSearchCV
6. **CNN fine-tuning** (EfficientNetV2-S end-to-end)
7. **Final evaluation** with full metrics (accuracy, precision, recall, F1, confusion matrix)
8. **Overfitting control** (< 5% difference between train and val)

### Imports

- **`numpy`** (`np`): library for numerical array/matrix operations. Used to manipulate images (which are pixel arrays) and labels.
- **`matplotlib.pyplot`** (`plt`): visualization library. Creates charts (histograms, bars, lines, images).
- **`pandas`** (`pd`): data table library (DataFrames). Used to display results in tabular format.
- **`PIL.Image`**: library to open and manipulate images.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

## 1. Load Raw Data

We load the `.npz` file (NumPy compressed format that stores multiple arrays).

- **`np.load()`**: opens the file and returns a dictionary-like object where each key is an array.
- **`data.keys()`**: shows which arrays are stored (X_train, y_train, X_val, y_val, X_test, y_test).
- **`.shape`**: array dimensions. E.g.: `(13036, 224, 224, 3)` = 13036 images of 224x224 pixels with 3 RGB channels.
- **`.dtype`**: data type. `uint8` = integers from 0 to 255 (pixel values).
- **`.min()` / `.max()`**: value range.

In [ ]:
data = np.load("../data/processed/waste_dataset_224.npz")

print("Keys:", list(data.keys()))
print()
for key in data.keys():
    arr = data[key]
    print(f"{key}: shape={arr.shape}, dtype={arr.dtype}, min={arr.min()}, max={arr.max()}")

We split the arrays into descriptive variables:

- **`X_train`, `X_val`, `X_test`**: images (features). `X` = model inputs.
- **`y_train`, `y_val`, `y_test`**: labels. `y` = what the model must predict. Integer numbers (0-7) representing classes.
- **`ORIGINAL_CLASSES`**: class names in alphabetical order (as saved by `image_dataset_from_directory` in Colab).
- **Train/Val/Test split**: train=80% (for training), val=10% (for tuning), test=10% (final evaluation, never touched during training).

In [ ]:
X_train_raw, y_train_raw = data["X_train"], data["y_train"]
X_val_raw, y_val_raw = data["X_val"], data["y_val"]
X_test_raw, y_test_raw = data["X_test"], data["y_test"]

ORIGINAL_CLASSES = ["cardboard", "glass", "metal", "organic", "paper", "plastic", "textile", "trash"]

print(f"Total images: {len(y_train_raw) + len(y_val_raw) + len(y_test_raw)}")
print(f"Image size: {X_train_raw.shape[1]}x{X_train_raw.shape[2]} px, {X_train_raw.shape[3]} channels (RGB)")
print(f"Pixel range: [{X_train_raw.min()}, {X_train_raw.max()}] (uint8)")
print(f"Classes ({len(ORIGINAL_CLASSES)}): {ORIGINAL_CLASSES}")

## 2. Exploratory Data Analysis (EDA)

### 2.1 Class Distribution

We visualize how many images there are per class in each split (train, val, test).

- **`plt.subplots(1, 3)`**: creates 1 row x 3 columns of charts (one per split).
- **`figsize=(18, 5)`**: chart size in inches (width x height).
- **`np.unique(y, return_counts=True)`**: finds unique values in `y` and counts how many times each appears.
- **`ax.bar(labels, counts)`**: bar chart. Each bar = one class, height = number of images.
- **`ax.bar_label(bars)`**: displays the exact number above each bar.
- **`ax.tick_params(axis='x', rotation=45)`**: rotates X-axis labels 45 degrees to avoid overlap.
- **`plt.suptitle()`**: overall chart title (super title).
- **`plt.tight_layout()`**: adjusts margins so text doesn't get cut off.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, y) in zip(axes, [("Train", y_train_raw), ("Val", y_val_raw), ("Test", y_test_raw)]):
    unique, counts = np.unique(y, return_counts=True)
    labels = [ORIGINAL_CLASSES[i] for i in unique]
    bars = ax.bar(labels, counts, color='steelblue')
    ax.bar_label(bars)
    ax.set_title(f"{name} ({len(y)} images)")
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=45)
plt.suptitle("Class Distribution (8 classes, before cleaning)", fontsize=14)
plt.tight_layout()
plt.show()

### 2.2 Sample Images per Class

We display one sample image per class to visually verify the data is correct.

- **`plt.subplots(2, 4)`**: 2 rows x 4 columns = 8 slots (one per class).
- **`axes.flat`**: converts the 2D axes matrix into a flat list for easy iteration.
- **`enumerate(ORIGINAL_CLASSES)`**: returns pairs (index, class_name).
- **`X_train_raw[y_train_raw == idx]`**: filters all images whose label is `idx` (boolean mask).
- **`[0]`**: takes the first image of that class.
- **`ax.imshow(img)`**: displays the image in the chart.
- **`ax.axis("off")`**: hides the axes (numbers) around the image.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (idx, cls) in zip(axes.flat, enumerate(ORIGINAL_CLASSES)):
    img = X_train_raw[y_train_raw == idx][0]
    ax.imshow(img)
    ax.set_title(f"{cls} (idx={idx})")
    ax.axis("off")
plt.suptitle("Sample Image per Class", fontsize=14)
plt.tight_layout()
plt.show()

### 2.3 Pixel Statistics

We analyze the pixel value distribution per color channel (RGB). This helps understand if the data needs special normalization.

- **`np.random.seed(42)`**: fixes the random seed for reproducibility (always selects the same 1000 images).
- **`np.random.choice(len(X), 1000, replace=False)`**: selects 1000 random indices without replacement.
- **`.astype(np.float32)`**: converts from uint8 (integer) to float32 (decimal) for statistics.
- **`sample[:, :, :, ch]`**: selects channel `ch` from all images. The 4 `:` mean: all images, all rows, all columns, specific channel.
- **`ax.hist()`**: histogram — shows the frequency of each pixel value.
- **`flatten()`**: converts a multidimensional array to 1D (required for the histogram).
- **`alpha=0.7`**: 70% transparency (0=invisible, 1=opaque).

In [ ]:
np.random.seed(42)
sample = X_train_raw[np.random.choice(len(X_train_raw), 1000, replace=False)].astype(np.float32)

print("Pixel statistics (1000 random train images):")
for ch, name in enumerate(["Red", "Green", "Blue"]):
    vals = sample[:, :, :, ch]
    print(f"  {name}: mean={vals.mean():.1f}, std={vals.std():.1f}, min={vals.min():.0f}, max={vals.max():.0f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, ch, color in zip(axes, range(3), ['red', 'green', 'blue']):
    ax.hist(sample[:, :, :, ch].flatten(), bins=50, color=color, alpha=0.7)
    ax.set_title(f"{color.capitalize()} Channel")
    ax.set_xlabel("Pixel Value")
plt.suptitle("Pixel Distribution per Channel", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Data Cleaning

### Class mapping applied in `data-raw-R.ipynb`

3 datasets were unified with the following mapping:

| Original class | Source | Unified class |
|---|---|---|
| cardboard | garbage_classification | cardboard |
| glass | garbage_classification | glass |
| metal | garbage_classification | metal |
| paper | garbage_classification | paper |
| plastic | garbage_classification | plastic |
| trash | garbage_classification | trash |
| **brown-glass** | garbage_12_classes | **glass** |
| **green-glass** | garbage_12_classes | **glass** |
| **white-glass** | garbage_12_classes | **glass** |
| **biological** | garbage_12_classes | **organic** |
| clothes | garbage_12_classes | textile |
| Cardboard | realwaste | cardboard |
| Glass | realwaste | glass |
| Metal | realwaste | metal |
| Paper | realwaste | paper |
| Plastic | realwaste | plastic |
| Miscellaneous Trash | realwaste | trash |
| **Food Organics** | realwaste | **organic** |
| **Vegetation** | realwaste | **organic** |
| Textile Trash | realwaste | textile |

**Skipped:** `battery`, `shoes`

**Removing now:** `textile` (index 6)

We remove the `textile` class (index 6) from the dataset and remap the indices.

- **`mask = y != idx`**: creates a boolean array with `True` where the label is NOT textile. E.g.: `[True, True, False, True]`.
- **`X[mask]`**: filters only the images where `mask=True` (removes textile images).
- **`.copy()`**: creates an independent copy to avoid modifying the original data.
- **`y_out[y_out > idx] -= 1`**: after removing textile (idx=6), trash was idx=7 and now must be idx=6. All indices greater than 6 are shifted down by 1.
- **Final distribution**: bar chart to visually verify the class balance after cleaning.

In [ ]:
TEXTILE_IDX = 6

def remove_class(X, y, idx):
    mask = y != idx
    X_out = X[mask]
    y_out = y[mask].copy()
    y_out[y_out > idx] -= 1
    return X_out, y_out

print("BEFORE:")
print(f"  Train: {X_train_raw.shape[0]}, Val: {X_val_raw.shape[0]}, Test: {X_test_raw.shape[0]}")
print(f"  Classes: {ORIGINAL_CLASSES}")

X_train, y_train = remove_class(X_train_raw, y_train_raw, TEXTILE_IDX)
X_val, y_val = remove_class(X_val_raw, y_val_raw, TEXTILE_IDX)
X_test, y_test = remove_class(X_test_raw, y_test_raw, TEXTILE_IDX)

CLASS_NAMES = ["cardboard", "glass", "metal", "organic", "paper", "plastic", "trash"]

print(f"\nAFTER:")
print(f"  Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")
print(f"  Removed: {X_train_raw.shape[0] - X_train.shape[0]} train, {X_val_raw.shape[0] - X_val.shape[0]} val, {X_test_raw.shape[0] - X_test.shape[0]} test")
print(f"  Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")

# Final distribution
fig, ax = plt.subplots(figsize=(10, 5))
unique, counts = np.unique(y_train, return_counts=True)
bars = ax.bar([CLASS_NAMES[i] for i in unique], counts, color='steelblue')
ax.bar_label(bars)
ax.set_title("Final Train Distribution (7 classes)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 4. Feature Extraction (EfficientNetV2-S Backbone)

**Transfer Learning**: instead of training a neural network from scratch (which would require millions of images), we use one already trained on ImageNet (14M images, 1000 categories). This network already "knows how to see" shapes, textures, colors, etc.

We use the backbone as a **feature extractor**: each image (224x224x3 = 150,528 values) is compressed into a vector of 1280 numbers representing its high-level visual characteristics. These vectors are the input for the sklearn models.

### Normalization

We convert pixels from range `[0, 255]` (integers) to `[0.0, 1.0]` (floats). Neural networks train better with small values because gradients remain stable.

- **`.astype(np.float32)`**: changes type from uint8 to float32 (needed for decimals).
- **`/ 255.0`**: divides each pixel by 255 (the maximum value). So 0→0.0, 128→0.502, 255→1.0.

In [ ]:
import tensorflow as tf

# Normalize pixels to [0, 1]
X_train_norm = X_train.astype(np.float32) / 255.0
X_val_norm = X_val.astype(np.float32) / 255.0
X_test_norm = X_test.astype(np.float32) / 255.0

print(f"Normalized range: [{X_train_norm.min()}, {X_train_norm.max()}]")

### Loading the EfficientNetV2-S backbone

- **`EfficientNetV2S`**: efficient and accurate neural network architecture (2021, Google).
- **`include_top=False`**: don't include the original classification head (1000 ImageNet classes). We only want the feature extractor.
- **`weights="imagenet"`**: load pre-trained weights from ImageNet.
- **`input_shape=(224, 224, 3)`**: expects 224x224 pixel images with 3 channels (RGB).
- **`pooling="avg"`**: applies GlobalAveragePooling at the end → converts feature maps from (7,7,1280) into a 1280-dimensional vector.
- **`trainable = False`**: freezes the weights. We only use the network as an extractor, not modifying it.

In [ ]:
# Load backbone
base_model = tf.keras.applications.EfficientNetV2S(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3),
    pooling="avg"  # GlobalAveragePooling -> output (batch, 1280)
)
base_model.trainable = False

print(f"Backbone output shape: {base_model.output_shape}")
print(f"Extracting features...")

### Feature extraction

We pass all images through the backbone to obtain 1280-dimensional vectors.

- **`model.predict(X, batch_size=32)`**: processes images in batches of 32 (not all at once, due to memory).
- **`verbose=1`**: shows progress bar.
- **`F_train`**: features from the training set. Shape: `(N_train, 1280)`.

These features are the input for the sklearn models (which cannot process images directly, but can handle numeric vectors).

In [ ]:
# Extract features (this takes a few minutes on CPU)
BATCH_SIZE = 32

def extract_features(X, model, batch_size=32):
    features = model.predict(X, batch_size=batch_size, verbose=1)
    return features

F_train = extract_features(X_train_norm, base_model, BATCH_SIZE)
F_val = extract_features(X_val_norm, base_model, BATCH_SIZE)
F_test = extract_features(X_test_norm, base_model, BATCH_SIZE)

print(f"\nFeature shapes:")
print(f"  F_train: {F_train.shape}")
print(f"  F_val:   {F_val.shape}")
print(f"  F_test:  {F_test.shape}")

### Save features

We save the extracted features to a separate `.npz` file. This way, if we want to retrain the sklearn models, we don't need to re-run the extraction (which takes several minutes).

- **`np.savez_compressed()`**: saves multiple arrays into a compressed `.npz` file.

In [ ]:
# Save features to avoid re-extracting
np.savez_compressed("../data/processed/waste_features_1280.npz",
    F_train=F_train, y_train=y_train,
    F_val=F_val, y_val=y_val,
    F_test=F_test, y_test=y_test)
print("Features saved to data/processed/waste_features_1280.npz")

## 5. Train Multiple Models (Scikit-learn)

We train 4 different models using the extracted features (1280-dimensional vectors, not images).

**Models:**
- **Random Forest**: ensemble of many decision trees that vote. `n_estimators=300` = 300 trees, `max_depth=20` = maximum depth per tree.
- **XGBoost**: gradient boosting — trees trained sequentially, each one correcting the errors of the previous one. `learning_rate=0.1` = how much each tree contributes.
- **SVM (RBF)**: Support Vector Machine with RBF kernel. Finds the hyperplane that best separates classes. `C=10` = error penalty, `gamma="scale"` = kernel parameter.
- **LightGBM**: similar to XGBoost but faster, uses histograms to find the best splits.

**Metrics computed:**
- **`accuracy_score()`**: % of correct predictions.
- **Train/Val/Test Acc**: accuracy on each split.
- **Overfit %**: `|Train Acc - Val Acc| × 100`. If > 5%, the model is memorizing instead of generalizing.
- **`n_jobs=-1`**: uses all CPU cores for parallelization.
- **`time.time()`**: measures training time in seconds.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import lightgbm as lgb
import time

# Optional: load features if already extracted
# feat = np.load("../data/processed/waste_features_1280.npz")
# F_train, y_train = feat["F_train"], feat["y_train"]
# F_val, y_val = feat["F_val"], feat["y_val"]
# F_test, y_test = feat["F_test"], feat["y_test"]

models = {
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1),
    "XGBoost": xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric="mlogloss"),
    "SVM (RBF)": SVC(kernel="rbf", C=10, gamma="scale", random_state=42),
    "LightGBM": lgb.LGBMClassifier(n_estimators=300, max_depth=10, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1),
}

results = []

for name, clf in models.items():
    print(f"\nTraining {name}...")
    start = time.time()
    clf.fit(F_train, y_train)
    train_time = time.time() - start
    
    train_acc = accuracy_score(y_train, clf.predict(F_train))
    val_acc = accuracy_score(y_val, clf.predict(F_val))
    test_acc = accuracy_score(y_test, clf.predict(F_test))
    overfitting = abs(train_acc - val_acc) * 100
    
    results.append({
        "Model": name,
        "Train Acc": f"{train_acc:.4f}",
        "Val Acc": f"{val_acc:.4f}",
        "Test Acc": f"{test_acc:.4f}",
        "Overfit %": f"{overfitting:.2f}%",
        "Time (s)": f"{train_time:.1f}"
    })
    print(f"  Train: {train_acc:.4f}, Val: {val_acc:.4f}, Test: {test_acc:.4f}, Overfit: {overfitting:.2f}%, Time: {train_time:.1f}s")

df_results = pd.DataFrame(results)
print("\n" + "="*70)
print(df_results.to_string(index=False))

## 6. Cross-Validation (StratifiedKFold)

**Cross-validation** evaluates model robustness by splitting data into K partitions (folds). Each fold is used once as validation and K-1 times as training.

- **`StratifiedKFold(n_splits=5)`**: splits data into 5 parts while maintaining the class proportion in each fold. "Stratified" = respects class balance.
- **`shuffle=True`**: shuffles data before splitting.
- **`np.concatenate([F_train, F_val])`**: joins train + val to have more data available for CV.
- **`cross_val_score()`**: trains the model K times with different splits and returns the score for each fold.
- **`scoring="accuracy"`**: metric to evaluate on each fold.
- **`scores.mean()` / `scores.std()`**: mean and standard deviation. A low std indicates the model is consistent.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Combine train + val for cross-validation
F_trainval = np.concatenate([F_train, F_val], axis=0)
y_trainval = np.concatenate([y_train, y_val], axis=0)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []
for name, clf in models.items():
    print(f"Cross-validating {name}...")
    scores = cross_val_score(clf, F_trainval, y_trainval, cv=skf, scoring="accuracy", n_jobs=-1)
    cv_results.append({
        "Model": name,
        "CV Mean": f"{scores.mean():.4f}",
        "CV Std": f"{scores.std():.4f}",
        "Folds": str([f"{s:.4f}" for s in scores])
    })
    print(f"  {scores.mean():.4f} +/- {scores.std():.4f}")

df_cv = pd.DataFrame(cv_results)
print("\n" + "="*70)
print(df_cv.to_string(index=False))

## 7. Hyperparameter Optimization (Best Model)

**Hyperparameter optimization** searches for the best parameter combination for the winning model.

- **`RandomizedSearchCV`**: tests N random parameter combinations (faster than `GridSearchCV` which tests ALL combinations).
- **`param_distributions`**: dictionary with the range of each hyperparameter:
  - `n_estimators`: number of trees.
  - `max_depth`: maximum depth (deeper = more complex = more overfitting risk).
  - `learning_rate`: learning step size.
  - `subsample`: fraction of data used per tree (< 1.0 = regularization).
  - `colsample_bytree`: fraction of features used per tree.
  - `min_child_weight`: minimum samples in a leaf node.
- **`n_iter=20`**: tests 20 random combinations.
- **`cv=skf`**: uses the same cross-validation strategy (5 stratified folds).
- **`search.best_score_`**: best score found.
- **`search.best_params_`**: the hyperparameters that achieved that score.
- **`search.best_estimator_`**: the model already trained with the best parameters.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 10, 15],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
}

print("Running RandomizedSearchCV on XGBoost...")
search = RandomizedSearchCV(
    xgb.XGBClassifier(random_state=42, n_jobs=-1, eval_metric="mlogloss"),
    param_distributions,
    n_iter=20,
    cv=skf,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search.fit(F_trainval, y_trainval)

print(f"\nBest score: {search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

best_sklearn = search.best_estimator_

## 8. Model Comparison

### Confusion matrices per model

The **confusion matrix** shows for each class how many predictions were correct and how many were confused with another class.
- Rows = actual class, Columns = predicted class.
- Diagonal = correct predictions. Off-diagonal = errors.
- Helps identify which classes get confused with each other (e.g., metal with trash).

- **`confusion_matrix(y_test, y_pred)`**: computes the matrix.
- **`ConfusionMatrixDisplay`**: renders it as a colored heatmap.
- **`cmap="Blues"`**: blue color palette (darker = more samples).
- **`values_format="d"`**: displays numbers as integers.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, (name, clf) in zip(axes.flat, models.items()):
    y_pred = clf.predict(F_test)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title(f"{name} (acc={acc:.4f})")

plt.suptitle("Confusion Matrix - All Models", fontsize=16)
plt.tight_layout()
plt.show()

### Visual accuracy comparison

Bar chart comparing the Test Accuracy of all models.

- **`ax.bar()`**: bar chart, different color for each model.
- **`ax.bar_label(bars, fmt="%.4f")`**: shows the value with 4 decimals above each bar.
- **`ax.set_ylim(min - 0.05, 1.0)`**: adjusts the Y-axis so differences between models are visible.

In [ ]:
model_names = [r["Model"] for r in results]
test_accs = [float(r["Test Acc"]) for r in results]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(model_names, test_accs, color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'])
ax.bar_label(bars, fmt="%.4f")
ax.set_ylabel("Test Accuracy")
ax.set_title("Model Comparison - Test Accuracy")
ax.set_ylim(min(test_accs) - 0.05, 1.0)
plt.tight_layout()
plt.show()

## 9. CNN Fine-tuning (EfficientNetV2-S End-to-End)

In addition to sklearn models, we train the full CNN end-to-end for comparison.

### CNN Architecture

1. **`Input(shape=(224,224,3))`**: input layer, defines the expected size.
2. **`base_model(inputs, training=False)`**: passes the image through EfficientNetV2-S. `training=False` keeps BatchNormalization in inference mode.
3. **`Dropout(0.3)`**: randomly turns off 30% of neurons. Prevents overfitting.
4. **`Dense(256, activation="relu")`**: dense layer with 256 neurons. `relu` = max(0, x), introduces non-linearity.
5. **`Dense(7, activation="softmax")`**: output layer. 7 neurons (one per class). `softmax` converts values into probabilities that sum to 1.0.

**Compilation:**
- **`Adam(lr=1e-3)`**: adaptive optimizer. `1e-3 = 0.001` = learning rate.
- **`sparse_categorical_crossentropy`**: loss function for multiclass classification with integer labels (not one-hot).
- **`metrics=["accuracy"]`**: metric to monitor during training.

In [ ]:
from tensorflow.keras import layers, Model

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation="softmax")(x)

cnn_model = Model(inputs, outputs)

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

### Phase 1: Frozen Backbone

Only the new layers (Dense + Dropout) are trained. The backbone is frozen.

- **`model.fit(X, y)`**: trains the model.
- **`epochs=20`**: maximum 20 full passes through the data.
- **`batch_size=32`**: processes 32 images at a time. Larger = faster but more memory.
- **`validation_data=(X_val, y_val)`**: evaluates on val after each epoch.
- **Callbacks** (automatic functions during training):
  - **`EarlyStopping(patience=3)`**: if val_loss doesn't improve for 3 epochs, stops training. `restore_best_weights=True` = reverts to the weights of the best epoch.
  - **`ReduceLROnPlateau(patience=2, factor=0.5)`**: if val_loss doesn't improve for 2 epochs, halves the learning rate.

In [ ]:
history_p1 = cnn_model.fit(
    X_train_norm, y_train,
    validation_data=(X_val_norm, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5)
    ]
)

### Phase 2: Fine-tuning (Unfreeze top 50 layers)

We unfreeze the last 50 layers of the backbone so they can adapt to our specific waste data.

- **`base_model.trainable = True`**: unfreezes all layers.
- **`layers[:-50]`**: selects all layers EXCEPT the last 50. These remain frozen.
- **`Adam(lr=1e-5)`**: very low learning rate (0.00001) to avoid destroying pre-trained weights. Small, careful adjustments.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print(f"Trainable params: {sum(p.numpy().size for p in cnn_model.trainable_weights):,}")

In [ ]:
history_p2 = cnn_model.fit(
    X_train_norm, y_train,
    validation_data=(X_val_norm, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5)
    ]
)

## 10. Final Evaluation & Metrics

### Full metrics

We evaluate the best sklearn model and the CNN on the test set (never-seen data).

- **`classification_report()`**: generates a table with metrics PER CLASS:
  - **Precision**: of the predictions as class X, how many were actually X? (avoid false positives)
  - **Recall**: of all actual samples of class X, how many were detected? (avoid false negatives)
  - **F1-score**: harmonic mean of precision and recall. A single number summarizing both.
  - **Support**: number of actual samples of that class in the test set.
- **`model.evaluate()`**: computes loss and accuracy of the keras model on a dataset.
- **`.argmax(axis=1)`**: for each image, selects the index with the highest probability as the prediction.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# ---- Best sklearn model ----
print("="*60)
print("BEST SKLEARN MODEL (Optimized)")
print("="*60)
y_pred_sk = best_sklearn.predict(F_test)
sk_acc = accuracy_score(y_test, y_pred_sk)
print(f"Test Accuracy: {sk_acc:.4f}")
print(classification_report(y_test, y_pred_sk, target_names=CLASS_NAMES))

# ---- CNN model ----
print("\n" + "="*60)
print("CNN (EfficientNetV2-S Fine-tuned)")
print("="*60)
cnn_loss, cnn_acc = cnn_model.evaluate(X_test_norm, y_test, verbose=0)
y_pred_cnn = cnn_model.predict(X_test_norm).argmax(axis=1)
print(f"Test Accuracy: {cnn_acc:.4f}")
print(classification_report(y_test, y_pred_cnn, target_names=CLASS_NAMES))

### Final comparison: Sklearn vs CNN

Side-by-side confusion matrices to visually compare where each approach succeeds and fails.

- **`plt.subplots(1, 2)`**: 2 charts side by side.
- **`cmap="Blues"` / `cmap="Greens"`**: different colors to distinguish the models.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

cm1 = confusion_matrix(y_test, y_pred_sk)
ConfusionMatrixDisplay(cm1, display_labels=CLASS_NAMES).plot(ax=ax1, cmap="Blues", values_format="d")
ax1.set_title(f"Best Sklearn ({sk_acc:.4f})")

cm2 = confusion_matrix(y_test, y_pred_cnn)
ConfusionMatrixDisplay(cm2, display_labels=CLASS_NAMES).plot(ax=ax2, cmap="Greens", values_format="d")
ax2.set_title(f"CNN Fine-tuned ({cnn_acc:.4f})")

plt.suptitle("Final Comparison: Best Sklearn vs CNN", fontsize=16)
plt.tight_layout()
plt.show()

### Overfitting Check

**Overfitting = difference between train accuracy and val accuracy.** If the model has 99% on train but 70% on val, it is memorizing data instead of learning general patterns. The project requirement is < 5%.

- **`abs(train - val) * 100`**: absolute difference as percentage.
- We compare both models (sklearn and CNN) to verify they both meet the requirement.

In [ ]:
sk_train_acc = accuracy_score(y_train, best_sklearn.predict(F_train))
sk_val_acc = accuracy_score(y_val, best_sklearn.predict(F_val))
sk_overfit = abs(sk_train_acc - sk_val_acc) * 100

cnn_train_acc = max(history_p1.history['accuracy'] + history_p2.history['accuracy'])
cnn_val_acc = max(history_p1.history['val_accuracy'] + history_p2.history['val_accuracy'])
cnn_overfit = abs(cnn_train_acc - cnn_val_acc) * 100

print(f"{'Model':<25} {'Train':>8} {'Val':>8} {'Test':>8} {'Overfit':>10}")
print("-" * 65)
print(f"{'Best Sklearn':<25} {sk_train_acc:>8.4f} {sk_val_acc:>8.4f} {sk_acc:>8.4f} {sk_overfit:>9.2f}%")
print(f"{'CNN Fine-tuned':<25} {cnn_train_acc:>8.4f} {cnn_val_acc:>8.4f} {cnn_acc:>8.4f} {cnn_overfit:>9.2f}%")
print()
print(f"Sklearn overfit < 5%: {'YES' if sk_overfit < 5 else 'NO'}")
print(f"CNN overfit < 5%:     {'YES' if cnn_overfit < 5 else 'NO'}")

### Training History (CNN)

Accuracy and loss chart across epochs for both phases. Helps visualize if the model is learning or overfitting.

- **`history.history['accuracy']`**: list with the accuracy of each epoch.
- **We concatenate Phase 1 + Phase 2** for the complete history.
- **`ax.plot()`**: line chart.
- **`ax.axvline()`**: vertical line marking where fine-tuning starts.
- **`linestyle="--"`**: dashed line.
- **`ax.legend()`**: shows the legend (which line is Train and which is Val).

**Signs of good training**: Train and Val go down together (loss) or up together (accuracy). If Train goes much higher than Val = overfitting.

In [ ]:
acc = history_p1.history['accuracy'] + history_p2.history['accuracy']
val_acc = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy']
loss = history_p1.history['loss'] + history_p2.history['loss']
val_loss = history_p1.history['val_loss'] + history_p2.history['val_loss']
p1_len = len(history_p1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(acc, label="Train")
ax1.plot(val_acc, label="Val")
ax1.axvline(x=p1_len-1, color="gray", linestyle="--", label="Fine-tune start")
ax1.set_title("Accuracy")
ax1.legend()

ax2.plot(loss, label="Train")
ax2.plot(val_loss, label="Val")
ax2.axvline(x=p1_len-1, color="gray", linestyle="--", label="Fine-tune start")
ax2.set_title("Loss")
ax2.legend()

plt.tight_layout()
plt.show()

## 11. Save Best Model

We save both trained models to use them in the application (Streamlit/Gradio).

- **`model.save("file.keras")`**: saves the full CNN (architecture + weights + optimizer) in Keras format.
- **`joblib.dump(model, "file.joblib")`**: saves the serialized sklearn model. `joblib` is more efficient than `pickle` for large arrays.
- **`np.save("class_names.npy")`**: saves the class name list so the app can map index to name.

In [ ]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

# Save CNN
cnn_model.save("../models/efficientnetv2s_waste.keras")
print("CNN saved: models/efficientnetv2s_waste.keras")

# Save best sklearn model
joblib.dump(best_sklearn, "../models/best_sklearn_waste.joblib")
print(f"Sklearn saved: models/best_sklearn_waste.joblib")

# Save class names
np.save("../models/class_names.npy", CLASS_NAMES)
print(f"Class names saved: models/class_names.npy")